# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:
# Task 1 — fossil fuel dependency by country

# Keep only Coal, Oil and Natural Gas.
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Calculate fossil share of total energy for each country and region.
country_fossil_share = (
    df_fossil.groupby('Country')['TWh'].sum()
    / df.groupby('Country')['TWh'].sum()
).sort_values(ascending=False)

region_fossil_share = (
    df_fossil.groupby('Region')['TWh'].sum()
    / df.groupby('Region')['TWh'].sum()
).sort_values(ascending=False)

top_country = country_fossil_share.index[0]
top_country_share = country_fossil_share.iloc[0]
top_region = region_fossil_share.index[0]
top_region_share = region_fossil_share.iloc[0]

# CVD-safe course palette: blue, orange and yellow.
fossil_colours = {
    'Coal': '#2E75B6',
    'Oil': '#E07B39',
    'Natural Gas': '#F5C842'
}

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colours,
    labels={'TWh': 'Energy (TWh)', 'Source': 'Fossil source'},
    title=(
        f'{top_country} has the highest fossil share among countries '
        f'({top_country_share:.1%}); {top_region} leads among regions '
        f'({top_region_share:.1%})'
    ),
    height=750,
    template='plotly_white'
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>'
)

# Treemap parent nodes (Region and Country) are grey; source leaves retain
# their assigned source colours.
fig.data[0].marker.colors = [
    colour if colour in fossil_colours.values() else '#DDDDDD'
    for colour in fig.data[0].marker.colors
]

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(t=90, l=15, r=15, b=15)
)

fig.show()

print(f'Most fossil-dependent country: {top_country} ({top_country_share:.1%})')
print(f'Most fossil-dependent region: {top_region} ({top_region_share:.1%})')


Most fossil-dependent country: Saudi Arabia (99.0%)
Most fossil-dependent region: Middle East (99.0%)


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [6]:
# Task 2 — low-carbon energy by country

# Keep Nuclear and Hydro, then aggregate their TWh by country.
df_low_carbon = df.loc[df['Source_Type'] == 'Low-carbon'].copy()
low_carbon_by_country = (
    df_low_carbon.groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)

# Treemap requires a root column for the requested two-level hierarchy.
low_carbon_by_country['All'] = 'Low-carbon'
leading_country = low_carbon_by_country.iloc[0]['Country']
leading_twh = low_carbon_by_country.iloc[0]['TWh']

# Treemap: area represents total low-carbon TWh.
fig_treemap = px.treemap(
    low_carbon_by_country,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    labels={'TWh': 'Low-carbon energy (TWh)'},
    title='Low-carbon energy by country (Nuclear + Hydro)',
    height=650,
    template='plotly_white'
)
fig_treemap.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>'
)
fig_treemap.update_layout(coloraxis_showscale=False)
fig_treemap.show()

# Horizontal bar chart: a shared baseline makes country ranking easier to compare.
fig_bar = px.bar(
    low_carbon_by_country,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    text='TWh',
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    title=f'{leading_country} leads in low-carbon energy with {leading_twh:,.0f} TWh',
    height=max(500, 25 * len(low_carbon_by_country)),
    template='plotly_white'
)
fig_bar.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside',
    cliponaxis=False
)
fig_bar.update_layout(
    yaxis=dict(autorange='reversed'),
    coloraxis_showscale=False,
    margin=dict(r=100)
)
fig_bar.show()

print(f'Leading country: {leading_country} ({leading_twh:,.0f} TWh)')


Leading country: France (7,593 TWh)


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The horizontal bar chart tells the ranking story more clearly. Every bar starts from the same zero baseline, so it is easy to compare lengths and see which country has more low-carbon energy. The treemap is useful for showing relative magnitude and part-to-whole structure, but comparing rectangle areas precisely is harder, especially for countries with similar values.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:
# Task 3 — regional fossil mix and coal dependence

# Reuse df_fossil created in Task 1. Aggregate fossil TWh by region and source.
regional_fossil = (
    df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh']
    .sum()
)

# Convert each source to a share of its region's total fossil energy.
regional_fossil['Region_Total'] = (
    regional_fossil.groupby('Region')['TWh'].transform('sum')
)
regional_fossil['Share'] = (
    regional_fossil['TWh'] / regional_fossil['Region_Total']
)

# Sort regions by coal share, highest first.
coal_shares = (
    regional_fossil.loc[regional_fossil['Source'] == 'Coal']
    .set_index('Region')['Share']
    .sort_values(ascending=False)
)
region_order = coal_shares.index.tolist()
most_coal_dependent_region = coal_shares.index[0]
most_coal_share = coal_shares.iloc[0]

# Show the coal percentage label only; Oil and Natural Gas remain unlabelled.
regional_fossil['Coal_Label'] = np.where(
    regional_fossil['Source'] == 'Coal',
    regional_fossil['Share'].map('{:.0%}'.format),
    ''
)

# Coal uses the same blue as Task 1; the other sources are grey.
regional_colours = {
    'Coal': '#2E75B6',
    'Oil': '#AAAAAA',
    'Natural Gas': '#DDDDDD'
}

fig = px.bar(
    regional_fossil,
    x='Share',
    y='Region',
    color='Source',
    text='Coal_Label',
    orientation='h',
    color_discrete_map=regional_colours,
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    labels={'Share': 'Share of regional fossil energy', 'Source': 'Source'},
    title=(
        f'{most_coal_dependent_region} has the highest coal share '
        f'of fossil energy ({most_coal_share:.1%})'
    ),
    height=max(450, 90 * len(region_order)),
    template='plotly_white'
)

fig.update_traces(
    textposition='inside',
    insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white',
    marker_line_width=1,
    hovertemplate=(
        '<b>%{y}</b><br>%{fullData.name}: %{x:.1%}'
        '<extra></extra>'
    )
)

fig.update_layout(
    barmode='stack',
    barnorm='percent',
    xaxis=dict(tickformat='.0%', range=[0, 1], title='Share of regional fossil energy'),
    yaxis=dict(title='', autorange='reversed'),
    legend=dict(orientation='h', x=0, y=1.08),
    font=dict(family='Arial', size=12),
    margin=dict(t=110)
)

fig.show()

print(
    f'Most coal-dependent region: {most_coal_dependent_region} '
    f'({most_coal_share:.1%} of its fossil energy is coal)'
)


Most coal-dependent region: Africa (78.0% of its fossil energy is coal)


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The Task 1 treemap answers: Where is the absolute amount of fossil energy concentrated, and how does it break down by region, country and source? It shows the hierarchy and relative TWh magnitude.

The 100% stacked bar chart answers: Which region has the highest coal share within its fossil energy mix? Because every region's bar has the same total length, it makes the composition and coal-share comparison clear, but it does not show which region consumes the greatest absolute amount of fossil energy.